In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

### Tools Creation & Binding

In [5]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

# Tool Creation
@tool
def multiply(a:int,b:int)->int:
    '''Given 2 numbers a and b  this tool return their product'''
    return a*b

In [7]:
print(multiply.invoke({'a':5,'b':4}))

20


In [9]:
# tool - Bind
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")
llm_with_bind = llm.bind_tools([multiply])

### Tool - Calling :-

In [10]:
llm_with_bind.invoke("Hi,How are you ?")

AIMessage(content="I'm functioning properly, thanks for asking. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 238, 'total_tokens': 255, 'completion_time': 0.029483272, 'completion_tokens_details': None, 'prompt_time': 0.0139557, 'prompt_tokens_details': None, 'queue_time': 0.04578689, 'total_time': 0.043438972}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf214-4d62-70e0-b141-98f6dffce270-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 238, 'output_tokens': 17, 'total_tokens': 255})

In [12]:
llm_with_bind.invoke("Multiply 10 with 6").tool_calls

[{'name': 'multiply',
  'args': {'a': 10, 'b': 6},
  'id': 'thygegr4s',
  'type': 'tool_call'}]

## Tool-Execution :-

In [14]:
query = HumanMessage('Multiply 10 with 6')
messages = [query]

In [16]:
result = llm_with_bind.invoke(messages)
messages.append(result)

In [17]:
messages

[HumanMessage(content='Multiply 10 with 6', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'p3sfeqfq2', 'function': {'arguments': '{"a":10,"b":6}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 238, 'total_tokens': 257, 'completion_time': 0.032761507, 'completion_tokens_details': None, 'prompt_time': 0.013990838, 'prompt_tokens_details': None, 'queue_time': 0.046082671, 'total_time': 0.046752345}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf21a-8dd5-7351-b07d-0fbfcc514c13-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 6}, 'id': 'p3sfeqfq2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 238, 'output_tokens': 19, 'total_tokens': 257})]

In [18]:
tool_result = multiply.invoke(result.tool_calls[0])

In [19]:
tool_result

ToolMessage(content='60', name='multiply', tool_call_id='p3sfeqfq2')

In [20]:
messages.append(tool_result)

In [22]:
llm_with_bind.invoke(messages).content

'The result of the multiplication is 60.'